In [1]:
import lzma
import numpy as np
import polars as pl
from scipy.sparse import csr_matrix

In [2]:
ratings = (
    pl.read_csv("../week1/movieLense-100k/ratings.csv")
    .select(["userId", "movieId", "rating"])
    .with_columns(
        pl.col("userId").cast(pl.Int32),
        pl.col("movieId").cast(pl.Int32),
        pl.col("rating").cast(pl.Float32),
    )
)

users = ratings["userId"].to_numpy()
movies = ratings["movieId"].to_numpy()
stars = ratings["rating"].to_numpy()

R_user = csr_matrix(
    (stars, (users, movies)),
    shape=(users.max() + 1, movies.max() + 1),
    dtype=np.float32,
)

R_movie = R_user.tocsc()

In [ ]:
df = pl.read_parquet("hf://datasets/krishnakamath/movielens-32m-movies-enriched/data/train-00000-of-00001.parquet")
# This is where I got the plot summary from.

In [5]:
TEXT_COLUMNS = ["title", "genres", "plot_summary"]

def build_movie_text_lookup(movie_df):
    cols = [c for c in TEXT_COLUMNS if c in movie_df.columns]

    return {
        int(row["movie_id"]): " ".join(str(row[c]) for c in cols if row[c] is not None and str(row[c]).strip())
        for row in movie_df.select(["movie_id", *cols]).iter_rows(named=True)
    }


movie_text = build_movie_text_lookup(df)

In [6]:
BINS = ["5-4", "4-3", "3-2", "2-1"]

def get_rating_bin(rating):
    if rating >= 4: return "5-4"
    if rating >= 3: return "4-3"
    if rating >= 2: return "3-2"
    if rating >= 1: return "2-1"
    return None

In [ ]:
def get_user_corpora(user_id, target_movie_id=None):
    start, end = R_user.indptr[user_id], R_user.indptr[user_id + 1]

    corpora = {b: [] for b in BINS}
    ratings_by_bin = {b: [] for b in BINS}

    for movie_id, rating in zip(R_user.indices[start:end], R_user.data[start:end]):
        movie_id = int(movie_id)
        rating = float(rating)

        if movie_id == target_movie_id:
            continue

        rating_bin = get_rating_bin(rating)
        text = movie_text.get(movie_id)

        if rating_bin is None or not text:
            continue

        corpora[rating_bin].append(text)
        ratings_by_bin[rating_bin].append(rating)

    corpora = {b: "\n".join(texts) for b, texts in corpora.items()}

    return corpora, ratings_by_bin

In [20]:
def compressed_size(text):
    return len(lzma.compress(text.encode("utf-8")))

In [21]:
def compression_scores(user_id, movie_id):
    target_text = movie_text.get(movie_id)

    if not target_text:
        return None

    corpora, ratings_by_bin = get_user_corpora(user_id, target_movie_id=movie_id)

    scores = {}

    for rating_bin, corpus in corpora.items():
        if not corpus:
            continue

        before = compressed_size(corpus)
        after = compressed_size(corpus + "\n" + target_text)

        scores[rating_bin] = after - before

    return scores, ratings_by_bin

In [22]:
def predict_rating(user_id, movie_id):
    result = compression_scores(user_id, movie_id)

    if result is None:
        return None

    scores, ratings_by_bin = result

    if not scores:
        return None

    predicted_bin = min(scores, key=scores.get)
    bin_ratings = ratings_by_bin[predicted_bin]

    predicted_rating = np.mean(bin_ratings) if bin_ratings else BIN_DEFAULTS[predicted_bin]

    return float(predicted_rating), predicted_bin, scores

In [23]:
prediction = predict_rating(1, 70)

print(prediction)

(3.0, '4-3', {'5-4': 148, '4-3': 140, '3-2': 156, '2-1': 204})


In [24]:
def evaluate_rmse(ratings, max_samples=None, seed=42):
    evaluation = ratings

    if max_samples is not None and max_samples < len(ratings):
        evaluation = ratings.sample(n=max_samples, seed=seed)

    results = []

    for row in evaluation.iter_rows(named=True):
        user_id = int(row["userId"])
        movie_id = int(row["movieId"])
        true_rating = float(row["rating"])

        prediction = predict_rating(user_id, movie_id)

        if prediction is None:
            continue

        predicted_rating, predicted_bin, scores = prediction

        results.append({
            "userId": user_id,
            "movieId": movie_id,
            "true_rating": true_rating,
            "predicted_rating": predicted_rating,
            "predicted_bin": predicted_bin,
        })

    results = pl.DataFrame(results)

    if len(results) == 0:
        return None, results

    errors = results["true_rating"].to_numpy() - results["predicted_rating"].to_numpy()
    rmse = np.sqrt(np.mean(errors ** 2))

    return rmse, results

In [25]:
rmse, results = evaluate_rmse(ratings, max_samples=1000)

print("RMSE:", rmse)
print("Predictions:", len(results))

RMSE: 1.4305195287046777
Predictions: 993


In [17]:
def user_mean_prediction(user_id, target_movie_id):
    start, end = R_user.indptr[user_id], R_user.indptr[user_id + 1]

    movie_ids = R_user.indices[start:end]
    user_ratings = R_user.data[start:end]

    remaining = user_ratings[movie_ids != target_movie_id]

    return float(np.mean(remaining)) if len(remaining) else ratings["rating"].mean()

In [18]:
baseline_errors = []

for row in results.iter_rows(named=True):
    pred = user_mean_prediction(row["userId"], row["movieId"])
    baseline_errors.append((row["true_rating"] - pred) ** 2)

print("User-mean RMSE:", np.sqrt(np.mean(baseline_errors)))

User-mean RMSE: 1.0215839974328575
